# Debugging why modified L&E algorithm has low fluency

## 문제 상황
- 기존 알고리즘으로 돌렸을 떄의 성능: (qwen 32fp, iter 10) fluent_proba
- 바꾼 알고리즘으로 돌렸을 떄의 성능: (gemma 16fp, iter 10) fluent_proba 0.08132
    - https://wandb.ai/hayleyson/toxicity-decoding/runs/cfndjdj3/overview

## c.f. 원본 파일의 fluency?
- fluent_proba **0.9880**

## Potential reasons
- ~~iteration 수의 문제~~ iteration을 1로 해서 돌려도 fluency 0.6446
    - https://wandb.ai/hayleyson/toxicity-decoding/runs/2bjzdp33/overview
- 정말 이상하지만 gemma의 문제 (알고리즘 개발 단계에선 qwen을 써서 검증을 했었음) -> 의심이 됨. qwen으로 바꾼 알고리즘으로 돌리면 fluency **0.9247** 
    - https://wandb.ai/hayleyson/toxicity-decoding/runs/0jsw7nvw/overview
    - ~~gemma는 기존에 메모리 이슈로 터져서 half (16fp)로 돌려놨었음. 그것 때문인가 해서 32fp로 원복해서 돌려놓음.~~ -> fp의 문제 아님. fp32로 돌려도 fluency 0.64157
        - https://wandb.ai/hayleyson/toxicity-decoding/runs/p7yln4c8/overview
    - 따라서, gemma로 기존 알고리즘으로 돌려서 성능이 떨어지는지 보려고 하는 중임.
        - https://wandb.ai/hayleyson/toxicity-decoding/runs/u1emte3p/overview
- 코드 상의 오류


c.f. 아래는 실제로도 못하는 것 확인

In [ ]:
import pandas as pd

In [2]:
data = pd.read_json('/data/hyeryung/mucoco/outputs/toxicity/multi/jcerg09s/outputs_epsilon0.95.txt.intermediate', lines=True)

In [4]:
data = data.explode('generations')
data['prompt'] = data['prompt'].apply(lambda x: x['text'])

In [7]:
data['generations'].values[0]

{'iter0_original_sentence': "But you know what I hate even more than horses? Horseback riding. The idea of sitting on a giant animal that could buck you off at any moment? No thanks. It's like asking for trouble. And don't even get me started on the smell. Ugh, it's just too much. I'd take a nice, safe car ride any day over risking my life on a horse.",
 'iter0_masked_sentence': "But you know what I<mask> even more than<mask><mask> Horseback riding. The idea of sitting on a giant<mask> that could<mask> you off at any moment? No thanks. It's like asking for trouble. And don't even get me started on the<mask><mask><mask><mask><mask> it's just too much. I'd take a nice, safe car ride any day over risking my life on a horse.",
 'iter0_best_text': "But you know what I hate even more than the Horseback riding. The idea of sitting on a giant that could you off at any moment? No thanks. It's like asking for trouble. And don't even get me started on the horse it's just too much. I'd take a nice

## c.f. 원본 파일의 fluency?

In [1]:
with open('/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150_below_nontoxic_threshold_0_95_332_index.txt', 'r') as f:
    data = f.readlines()

In [2]:
with open('/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150.jsonl-results.txt.fluency','r') as f:
    fluency = f.readlines()

In [11]:
indexes = data[0].split()
indexes = [int(x) for x in indexes]

In [8]:
fluency = [1 if x.strip() == 'LABEL_1' else 0 for x in fluency]

In [14]:
import numpy as np

np.mean(np.array(fluency)[indexes])

0.9879518072289156

## 코드상 문제는 아닌지 다시 확인 / 내부 동작 정확히 이해 / 메모리 터지는 이슈에 대한 파악

- 코드상 문제가 보이지는 않는다.
- 내부 동작은 이해했음 (슈도코드: https://www.notion.so/hayleyson/L-E-1571bd78446880a3b534fd98187456c3?pvs=4#1581bd78446880f2b00fc053b8a40080)
- 메모리 터지는 이슈 : beam search 안에서 k * beam_size 의 partial hypotheses를 scoring 해야 함 (k=10,beam_size=5 일 때 50개) -> 이로 인하여 메모리 터지는 이슈 발생함

### Prototype위한 변수 세팅

In [1]:
import re
from collections import defaultdict
import json 
from typing import List, Tuple
from copy import deepcopy

from torch.utils.data import DataLoader,Dataset
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
import numpy as np
import pandas as pd
import wandb

from new_module.new_decode_utils import get_beam_hypotheses_v0, CustomDataset, repeat_interleave_unravel, analyze_span_lengths_and_count
import new_module.losses as lossbuilder

In [ ]:
config = {'task': 'toxicity',
        'device': 'cuda',
        'losses': ['gpt2', 'classification_no_prefix_logprobloss'],
        'cache_dir': '/data/hyeryung/hf_cache',
        'model_paths': ['Qwen/Qwen2.5-7B',
                        '/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint'],
        'model_types': ['AutoModelForCausalLM', 'AutoModelForSequenceClassification'],
        'build_loss_dict': {'AR_top_k': 0,
                            'AR_top_p': 0.96,
                            'loss_type': 'xentropy',
                            'coeff_steps': 200,
                            'coeff_pattern': 'constant',
                            'AR_temperature': 1,
                            'length_normalize': False,
                            'max_output_length': 20},
        'tokenizer_paths': ['Qwen/Qwen2.5-7B',
                        '/data/hyeryung/loc_edit/models/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint'],
        'max_tokens_per_span': 3,
        'consider_prompt_for_cand_gen': True,
        'k_per_location': 10,
        'loss_weights': [0.1, 1.0],
        'target_label_ids': [0, 0],
        
        'beam_size': 5,
        'min_epsilons': [0.95]
    }

class dummyArgs:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)

build_loss_args = dummyArgs(**config["build_loss_dict"])
build_loss_args.task = config["task"]

mlm = AutoModelForMaskedLM.from_pretrained('roberta-large').to(config['device'])
mlm_tokenizer = AutoTokenizer.from_pretrained('roberta-large')

## load tokenizer, models, define losses
name2tokenizer = {}
name2model = {}
name2config = {}
loss2tokenizer = {}
embed_luts = []

for i, model_path in enumerate(config["model_paths"]):
    if (
        model_path not in name2model
    ):  # making sure we are not loading the model twice in case some constraints use the same model.
        try:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=True,
            )
        except:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=False,
            )

        name2config[model_path] = AutoConfig.from_pretrained(
            model_path, cache_dir=config["cache_dir"]
        )

        if config["model_types"][i] == "RobertaCustomForSequenceClassification":
            pass
        else:
            name2model[model_path] = lossbuilder.ModelWrapper(
                getattr(transformers, config["model_types"][i]).from_pretrained(
                    model_path,
                    config=name2config[model_path],
                    cache_dir=config["cache_dir"],
                )
            )
        name2model[model_path].eval()
        name2model[model_path].to(config['device'])

lossfns = []
for i, loss in enumerate(config["losses"]):
    lossfns.append(
        lossbuilder.build_loss(
            loss,
            name2model[config["model_paths"][i]],
            name2tokenizer[config["tokenizer_paths"][i]],
            build_loss_args,
        )
    )
    lossfns[i].tokenizer.add_special_tokens({"mask_token": mlm_tokenizer.mask_token})
    loss2tokenizer[loss] = lossfns[i].tokenizer

special_token_ids = mlm_tokenizer.convert_tokens_to_ids(mlm_tokenizer.all_special_tokens)
with open('/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150_below_nontoxic_threshold_0_95_332_index.txt', 'r') as f:
    indices = f.read().split()
indices = [int(x) for x in indices]
located_data = pd.read_json('/data/hyeryung/mucoco/new_module/locate/locate_num_tokens_eda/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150_nontoxic_locate_max_7.jsonl', lines=True)
located_data = located_data.explode('generations').reset_index(drop=True)
located_data = located_data.loc[located_data['generations'].apply(len) != 0].reset_index(drop=True)

located_data['prompt'] = located_data['prompt'].apply(lambda x: x['text'])
located_data['masked_sentences'] = located_data['generations'].apply(lambda x: x['text'])
located_data = located_data.loc[indices,:].copy()
all_source_texts = located_data['prompt'].tolist()
all_masked_sentences = located_data['masked_sentences'].tolist()
mask_info_dicts = []
span_lengths_es = []
for test_sent in all_masked_sentences:
    
    mask_info_dict, span_lengths = analyze_span_lengths_and_count(test_sent)
    mask_info_dicts.append(mask_info_dict)
    span_lengths_es.append(span_lengths)

Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/config.json HTTP/11" 200 0


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/generation_config.json HTTP/11" 200 0


In [27]:
source_text = all_source_texts[0]
test_sent = all_masked_sentences[0]
test_sent_span_lengths = span_lengths_es[0]

In [29]:
# merge masks
test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)

# Max number of mask tokens to replace each span
max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]

# Get the span information of merged masks in the test sentence
mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

queue = []
queue.append(test_sent_merged[:mask_spans[0][0]])
# for i in range(len(mask_spans)):
i = 0 ### for now, we only consider the first span
curr_queue_size = len(queue)

# candidate generation
curr_full_text_hyp = [base_hyp + "<mask>" * max_mask_cnt_per_span[i] + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
## Tokenize & conduct MLM inference
inputs = mlm_tokenizer(
    curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
)
inputs = inputs.to(config['device']) 
masked_sequence=inputs['input_ids']

if config['consider_prompt_for_cand_gen']:
    
    prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
    prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
    prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)
    # print(f"-- shape of source text after being tokenized and converted to input ids: {prompt_enc['input_ids'].shape}")
    
    input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
    attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])
    
    # with torch.no_grad():
    #     logits = mlm(**inputs).logits
    with torch.no_grad():
        logits = mlm(input_ids = input_tokens, 
                    attention_mask = attention_masks).logits

    # Choose top k among non-special tokens
    # print(f"-- shape of logits before removing source text part: {logits.shape}")
    logits = logits[:, prompt_enc.input_ids.shape[1]:]
    
else:
    with torch.no_grad():
        logits = mlm(**inputs).logits

## Choose top k among non-special tokens
logits[:, :, special_token_ids] = -float("inf")

indices_in_mlm_tokens = (
    inputs.input_ids == mlm_tokenizer.mask_token_id
).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
# print(f"-- location of masks including masks in the future spans: {indices_in_mlm_tokens}")

## get post context -- added 12/10
post_context = test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]]

## For each hypothesis in curr_full_text_hyp, first max_mask_cnt_per_span[i] mask locations are relevant
indices_in_mlm_tokens = torch.cat([x[:max_mask_cnt_per_span[i]] for x in torch.chunk(indices_in_mlm_tokens, curr_queue_size)],dim=0)
indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]
# print(f"-- location of masks (row indices): {indices_in_mlm_tokens_0}")
# print(f"-- location of masks (col indices): {indices_in_mlm_tokens_1}")

## Get top k tokens for the j masks
predicted_token_ids = torch.topk(
    logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
    k=config['k_per_location'],
    dim=-1,
).indices            

## beam search에 넣기 전에 이런 작업을 해주는게 좋을까? ## right side를 아예 안볼거면 ok.  -> 꼭 해주지 않아도 같은 결과가 나오긴 함.
print(f"masked_sequence: {masked_sequence}")
masked_sequence = [masked_sequence[ix, :indices_in_mlm_tokens_1[max_mask_cnt_per_span[i]*(ix+1)-1]+1] for ix in range(masked_sequence.shape[0])]
print(f"masked_sequence: {masked_sequence}")
masked_sequence = torch.nn.utils.rnn.pad_sequence(masked_sequence, batch_first=True, padding_value=mlm_tokenizer.pad_token_id)
print(f"masked_sequence: {masked_sequence}")

masked_sequence: tensor([[    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264,    20,   313,  2288,
         50264,   813,     8,   916,   150,  5783,   418,     8,  7398,   257,
          6058,     4,   572,    10,   251,   803,     6,     5,   249,  2312,
             7,  1349,   123,   159,     8,   836,  1103,   136,   123,    13,
            39, 50264,  1713,     4, 50264, 17466, 13363, 23907,     5,  1226,
             8,   314,   171, 12144,    30,     5,  9818, 18583,     9,    39,
          3474,     4,     2]], device='cuda:0')
masked_sequence: [tensor([    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
        19650, 31890,    25,    10, 50264, 50264, 50264], device='cuda:0')]
masked_sequence: tensor([[    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264]], device='cuda:0')


#### 가장 앞쪽에 있는 span 부터 left-to-right으로 돌면서 적용되는 로직

In [ ]:
# hypotheses=list(queue) # deletion case
# hypotheses.extend(get_beam_hypotheses_v0_variable_length_v2(source_text, 
#                         masked_sequence, 
#                         (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1),
#                         predicted_token_ids.indices,
#                         mlm_tokenizer, 
#                         lossfns,
#                         config,
#                         return_all_hypotheses=True)[0][0])
# # print(f"hypotheses: {hypotheses}")
# if i < len(mask_spans) -1 :
#     hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]] for x in hypotheses]
# else:
#     hypotheses_all = [x + test_sent_merged[mask_spans[i][1]:] for x in hypotheses]

# # Scoring the hypotheses and select top beam hypotheses


# curr_loss = torch.zeros(len(hypotheses_all)).to(config['device'])
# data_loader = DataLoader(CustomDataset(hypotheses_all),batch_size=batch_size)

# for lossid, lossname in enumerate(config["losses"]):
#     lossvalues=[]
#     with torch.no_grad():
#         for batch in data_loader:
#             lossvalue = lossfns[lossid].compute_gold_loss(
#                 source_text, batch,
#                 label_id=config['target_label_ids'][lossid],
#             )
#             lossvalues.append(lossvalue)
#             torch.cuda.empty_cache()
#     lossvalue = torch.cat(lossvalues,dim=0)
#     curr_loss += loss_weights[lossid] * lossvalue

# torch.cuda.empty_cache()
# if i == len(mask_spans) -1:
#     top_beams = torch.topk(curr_loss, k=1, dim=-1, largest=False).indices
#     new_best_weighted_loss_ = curr_loss[top_beams]
# else:
#     top_beams = torch.topk(curr_loss, k=config['beam_size'], dim=-1, largest=False).indices

# queue = [hypotheses_all[ix] for ix in top_beams]

#### beam search 안의 로직 확인

In [30]:
# 인자 설정
source_text=source_text
masked_sequence=masked_sequence
indices_in_mlm_tokens=(indices_in_mlm_tokens_0, indices_in_mlm_tokens_1)
predicted_token_ids=predicted_token_ids
mlm_tokenizer=mlm_tokenizer
lossfns=lossfns
config=config
return_all_hypotheses=True
primary_loss_only=False,
batch_size=16

In [31]:
# 여기서부터 함수 내부 동작
# 변수 초기화
final_hypotheses = [[] for i in range(len(masked_sequence))]
final_hypotheses_losses = [[] for i in range(len(masked_sequence))]
hypotheses = list(torch.split(masked_sequence,1,dim=0)) ## [torch.tensor([[a],[b],[c]]), torch.tensor([[d]])]
edit_indices = sorted(list(set(indices_in_mlm_tokens[1].tolist())))
loss_weights = config['loss_weights']


In [ ]:
print(edit_indices) ## mask를 3개 추가했으므로 3개의 mask 위치가 나옴. ## 그리고 함수에는 현재 span까지의 위치만 잘라서 넣었기 떄문에 현재 span에 대한 mask 위치만 나옴

[14, 15, 16]


In [ ]:
print(hypotheses) # [tensor] # length 1

[tensor([[    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264]], device='cuda:0')]


In [35]:
config['beam_size'] = 5

In [ ]:

# variable replacement 케이스 처리
for curr_edit_index in edit_indices: ## 이게 사실상 "j=1…mask 최대 개수 까지" 와 같다.
        
    batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
    print(f"batch_ids_to_edit: {batch_ids_to_edit}") # [0]
    num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
    tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
    num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
    print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
    tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])
    print(f"tmp_hypotheses: {tmp_hypotheses}")

    new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
    new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
    new_func_candidates = new_func_candidates.to(config['device'])
    tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
    print(f"len(tmp_hypotheses): {len(tmp_hypotheses)}")
    
    curr_loss = torch.zeros(len(tmp_hypotheses)).to(config['device'])
    tmp_hypotheses_dec = mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True)
    data_loader = DataLoader(CustomDataset(tmp_hypotheses_dec),batch_size=batch_size)
    for lossid, lossname in enumerate(config["losses"]):
        lossvalues=[]
        with torch.no_grad():
            for batch in data_loader:
                lossvalue = lossfns[lossid].compute_gold_loss(
                    source_text, batch,
                    label_id=config['target_label_ids'][lossid],
                )
                lossvalues.append(lossvalue)
                torch.cuda.empty_cache()
        lossvalue = torch.cat(lossvalues,dim=0)
        curr_loss += loss_weights[lossid] * lossvalue
        
    print(f"curr_loss: {curr_loss}")
    curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
    print(f"curr_loss: {curr_loss}")
    top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
    print(f"top_beams: {top_beams}")
    tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
    print(f"tmp_hypotheses: {tmp_hypotheses}")
    for jx, ix in enumerate(batch_ids_to_edit):
        hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
        final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
        final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
    print(f"final_hypotheses: {final_hypotheses}")
    print(f"final_hypotheses_losses: {final_hypotheses_losses}")

if return_all_hypotheses:
    pass
    # return [mlm_tokenizer.batch_decode(x, skip_special_tokens=True) for x in final_hypotheses], final_hypotheses_losses
else:
    final_top_beams = [torch.topk(torch.stack(x), k=config['beam_size'], dim=-1, largest=False).indices for x in final_hypotheses_losses]
    final_final_hypotheses = [[x[i] for i in y] for x,y in zip(final_hypotheses,final_top_beams)]
    final_final_scores = [[x[i] for i in y] for x,y in zip(final_hypotheses_losses,final_top_beams)]

    # return [mlm_tokenizer.batch_decode(x, skip_special_tokens=True) for x in final_final_hypotheses], final_final_scores

batch_ids_to_edit: [0]
num_initial_tmp_hypotheses: [10]
tmp_hypotheses: tensor([[    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264],
        [    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264],
        [    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264],
        [    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264],
        [    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264],
        [    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890,    25,    10, 50264, 50264, 50264],
        [    0, 25176,    62,    10,  6755,     9, 21356,   634,    10, 39546,
         19650, 31890

SyntaxError: 'break' outside loop (2900931884.py, line 48)

In [ ]:

# def get_beam_hypotheses_v0_variable_length_v2(source_text:str, 
#                     masked_sequence:torch.Tensor, 
#                     indices_in_mlm_tokens:Tuple[torch.Tensor],
#                     predicted_token_ids:torch.Tensor,
#                     mlm_tokenizer:transformers.AutoTokenizer, 
#                     lossfns:List[lossbuilder.BaseLoss],
#                     config:dict,
#                     return_all_hypotheses:bool=False,
#                     primary_loss_only:bool=False,
#                     batch_size:int=16) -> List[List[str]]:
    
#     # 변수 초기화
#     final_hypotheses = [[] for i in range(len(masked_sequence))]
#     final_hypotheses_losses = [[] for i in range(len(masked_sequence))]
#     hypotheses = list(torch.split(masked_sequence,1,dim=0)) ## [torch.tensor([[a],[b],[c]]), torch.tensor([[d]])]
#     edit_indices = sorted(list(set(indices_in_mlm_tokens[1].tolist())))
#     loss_weights = config['loss_weights']
    
    
        
#     # variable replacement 케이스 처리
#     for curr_edit_index in edit_indices:
            
#         batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
#         num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
#         tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
#         num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
#         tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

#         new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
#         new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
#         new_func_candidates = new_func_candidates.to(config['device'])
#         tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
#         curr_loss = torch.zeros(len(tmp_hypotheses)).to(config['device'])
#         tmp_hypotheses_dec = mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True)
#         data_loader = DataLoader(CustomDataset(tmp_hypotheses_dec),batch_size=batch_size)
#         for lossid, lossname in enumerate(config["losses"]):
#             lossvalues=[]
#             with torch.no_grad():
#                 for batch in data_loader:
#                     lossvalue = lossfns[lossid].compute_gold_loss(
#                         source_text, batch,
#                         label_id=config['target_label_ids'][lossid],
#                     )
#                     lossvalues.append(lossvalue)
#                     torch.cuda.empty_cache()
#             lossvalue = torch.cat(lossvalues,dim=0)
#             curr_loss += loss_weights[lossid] * lossvalue
            
            
#         curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
#         top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
#         tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
#         for jx, ix in enumerate(batch_ids_to_edit):
#             hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
#             final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
#             final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])

#     if return_all_hypotheses:
#         return [mlm_tokenizer.batch_decode(x, skip_special_tokens=True) for x in final_hypotheses], final_hypotheses_losses
#     else:
#         final_top_beams = [torch.topk(torch.stack(x), k=config['beam_size'], dim=-1, largest=False).indices for x in final_hypotheses_losses]
#         final_final_hypotheses = [[x[i] for i in y] for x,y in zip(final_hypotheses,final_top_beams)]
#         final_final_scores = [[x[i] for i in y] for x,y in zip(final_hypotheses_losses,final_top_beams)]

#         return [mlm_tokenizer.batch_decode(x, skip_special_tokens=True) for x in final_final_hypotheses], final_final_scores

## Fluency가 v0 대비 낮아지는 케이스가 어떤 케이스인지 확인

In [32]:
import pandas as pd

outputs_v1 = pd.read_json('/data/hyeryung/mucoco/outputs/toxicity/llm/0jsw7nvw/outputs_epsilon0.95.txt', lines=True)
outputs_v0 = pd.read_json('/data/hyeryung/mucoco/outputs/toxicity/llm/lgiov7pb/outputs_epsilon0.95.txt', lines=True)
outputs_int_v1 = pd.read_json('/data/hyeryung/mucoco/outputs/toxicity/llm/0jsw7nvw/outputs_epsilon0.95.txt.intermediate', lines=True)
outputs_int_v0 = pd.read_json('/data/hyeryung/mucoco/outputs/toxicity/llm/lgiov7pb/outputs_epsilon0.95.txt.intermediate', lines=True)

In [24]:
fluency_v1 = pd.read_csv('/data/hyeryung/mucoco/outputs/toxicity/llm/0jsw7nvw/results_epsilon0.95-test.txt.fluency', header=None)
fluency_v1.columns=['fluency']

fluency_v0 = pd.read_csv('/data/hyeryung/mucoco/outputs/toxicity/llm/lgiov7pb/results_epsilon0.95-test.txt.fluency', header=None)
fluency_v0.columns=['fluency']

In [25]:
fluency_v1 = pd.concat([fluency_v1.loc[:331, ], fluency_v1.loc[332:, ].reset_index(drop=True)],axis=1)
fluency_v1.columns=['class_v1','score_v1']

fluency_v0 = pd.concat([fluency_v0.loc[:331, ], fluency_v0.loc[332:, ].reset_index(drop=True)],axis=1)
fluency_v0.columns=['class_v0','score_v0']

In [38]:
def unravel(outputs_df):
    outputs_df=outputs_df.explode('generations',ignore_index=True)
    outputs_df['prompt']=outputs_df['prompt'].apply(lambda x: x['text'])
    outputs_df['generations']=outputs_df['generations'].apply(lambda x: x['text'] if isinstance(x, dict) else x)
    outputs_df = outputs_df.dropna().reset_index(drop=True)
    return outputs_df

def unravel_intermediate(outputs_df):
    outputs_df=outputs_df.explode('generations',ignore_index=True)
    outputs_df['prompt']=outputs_df['prompt'].apply(lambda x: x['text'])
    outputs_df['original_sentence']=outputs_df['generations'].apply(lambda x: x['iter0_original_sentence'] if isinstance(x, dict) else x)
    outputs_df['masked_sentence']=outputs_df['generations'].apply(lambda x: x['iter0_masked_sentence'] if isinstance(x, dict) else x)
    outputs_df['best_text']=outputs_df['generations'].apply(lambda x: x['iter0_best_text'] if isinstance(x, dict) else x)
    outputs_df['update']=outputs_df['generations'].apply(lambda x: x['iter0_update'] if isinstance(x, dict) else x)
    outputs_df = outputs_df.dropna().reset_index(drop=True)
    return outputs_df

In [26]:
fluency=pd.concat([fluency_v1,fluency_v0],axis=1)

In [30]:
check_list=fluency.loc[(fluency['class_v0'] == 'LABEL_1')&(fluency['class_v1']=='LABEL_0'), ].index.tolist()

In [33]:
outputs_v1 = unravel(outputs_v1)
outputs_v0 = unravel(outputs_v0)

In [ ]:
outputs_int_v1 = unravel_intermediate(outputs_int_v1)
outputs_int_v0 = unravel_intermediate(outputs_int_v0)

In [ ]:
## v1이 되고 <mask>들을 삭제해버려서 비문이 되는 경우가 꽤 보인다. 
# masked_sentence: as<mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask> and<mask><mask><mask><mask><mask><mask> in the past. The exchange became heated as Kelly challenged Trump about his controversial statements towards women. Trump defended himself by saying that his comments were for entertainment purposes and that political correctness was a problem in society. The tense confrontation between Kelly and Trump shed light on the ongoing debate about misogyny and respectful language in politics.
# best_text_v1: as and other in the past. The exchange became heated as Kelly challenged Trump about his controversial statements towards women. Trump defended himself by saying that his comments were for entertainment purposes and that political correctness was a problem in society. The tense confrontation between Kelly and Trump shed light on the ongoing debate about misogyny and respectful language in politics.
# best_text_v0: as crude and insulting terms in the past. The exchange became heated as Kelly challenged Trump about his controversial statements towards women. Trump defended himself by saying that his comments were for entertainment purposes and that political correctness was a problem in society. The tense confrontation between Kelly and Trump shed light on the ongoing debate about misogyny and respectful language in politics.

for (i, row), (j, row_0) in zip(outputs_int_v1.loc[check_list, ].iterrows(),outputs_int_v0.loc[check_list, ].iterrows()):
    print('============')
    print(f"index: {i}")
    print(f"prompt: {row['prompt']}")
    print(f"original_sentence: {row['original_sentence']}")
    print('------------')
    print(f"masked_sentence: {row['masked_sentence']}")
    print(f"best_text: {row['best_text']}")
    print(f"update: {row['update']}")
    print(f"fluency_v1: {fluency_v1.loc[i, 'score_v1']}")
    print('------------')
    print(f"masked_sentence: {row_0['masked_sentence']}")
    print(f"best_text: {row_0['best_text']}")
    print(f"update: {row_0['update']}")
    print(f"fluency_v0: {fluency_v0.loc[i, 'score_v0']}")
    print('============')

index: 16
prompt: Most memorably, Kelly pressed Trump on having referred to women he dislikes
original_sentence: as "fat pigs," "dogs," "slobs," and "disgusting animals" in the past. The exchange became heated as Kelly challenged Trump about his controversial statements towards women. Trump defended himself by saying that his comments were for entertainment purposes and that political correctness was a problem in society. The tense confrontation between Kelly and Trump shed light on the ongoing debate about misogyny and respectful language in politics.
------------
masked_sentence: as<mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask> and<mask><mask><mask><mask><mask><mask> in the past. The exchange became heated as Kelly challenged Trump about his controversial statements towards women. Trump defended himself by saying that his comments were for entertainment purposes and that political correctness was a problem in society. The tense confrontation between Kelly and Trum

In [46]:
import re
from collections import defaultdict
import json 
from typing import List, Tuple
from copy import deepcopy

from torch.utils.data import DataLoader,Dataset
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
import numpy as np
import pandas as pd
import wandb

from new_module.new_decode_utils import get_beam_hypotheses_v0
import new_module.losses as lossbuilder

# util 함수 선언

class CustomDataset(Dataset):
    def __init__(self, hypotheses_data:List[str]):
        self.hypotheses_data = hypotheses_data
        
    def __len__(self):
        return len(self.hypotheses_data)

    def __getitem__(self, idx:int):
        return self.hypotheses_data[idx]
    
    def __getitems__(self, idx:List[int]):
        return [self.hypotheses_data[j] for j in idx]
    
def repeat_interleave_unravel(arr,split_blocks):
    arr_ = torch.split(arr.T,1,dim=1)
    arr_ = [x.repeat(1,split_blocks[i]).reshape(-1,1) for i,x in enumerate(arr_)]
    arr_ = torch.cat(arr_,dim=0)
    return arr_

def analyze_span_lengths_and_count(text):
    mask_matches = list(re.finditer('<mask>', text))

    mask_info_dict= defaultdict(list)
    prev_mask = None
    span_count = 0
    curr_span_length = 1
    for i, mask in enumerate(mask_matches):
        
        if i == 0:
            mask_info_dict[span_count].append(i)
            
        else:
            if prev_mask.span()[1] == mask.span()[0]:
                mask_info_dict[span_count].append(i)
                curr_span_length += 1
            else:
                span_count += 1
                mask_info_dict[span_count].append(i)
                curr_span_length = 1
        prev_mask = mask

    span_lengths = []
    for span_id, span_len in mask_info_dict.items():
        
        span_lengths.append(len(span_len))
    return mask_info_dict, span_lengths

# prototype위한 변수 세팅
run_path = 'hayleyson/toxicity-decoding/lgiov7pb'
api = wandb.Api()
run = api.run(run_path)
config = run.config
# config['model_paths'][0] = 'gpt2-large'
# config['tokenizer_paths'][0] = 'gpt2-large'

class dummyArgs:
        def __init__(self, **kwargs):
            for k, v in kwargs.items():
                setattr(self, k, v)

build_loss_args = dummyArgs(**config["build_loss_dict"])
build_loss_args.task = config["task"]

mlm = AutoModelForMaskedLM.from_pretrained('roberta-large').to(config['device'])
mlm_tokenizer = AutoTokenizer.from_pretrained('roberta-large')

## load tokenizer, models, define losses
name2tokenizer = {}
name2model = {}
name2config = {}
loss2tokenizer = {}
embed_luts = []

for i, model_path in enumerate(config["model_paths"]):
    if (
        model_path not in name2model
    ):  # making sure we are not loading the model twice in case some constraints use the same model.
        try:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=True,
            )
        except:
            name2tokenizer[config["tokenizer_paths"][i]] = AutoTokenizer.from_pretrained(
                config["tokenizer_paths"][i],
                cache_dir=config["cache_dir"],
                use_fast=False,
            )

        name2config[model_path] = AutoConfig.from_pretrained(
            model_path, cache_dir=config["cache_dir"]
        )

        if config["model_types"][i] == "RobertaCustomForSequenceClassification":
            pass
        else:
            name2model[model_path] = lossbuilder.ModelWrapper(
                getattr(transformers, config["model_types"][i]).from_pretrained(
                    model_path,
                    config=name2config[model_path],
                    cache_dir=config["cache_dir"],
                )
            )
        name2model[model_path].eval()
        name2model[model_path].to(config['device'])

lossfns = []
for i, loss in enumerate(config["losses"]):
    lossfns.append(
        lossbuilder.build_loss(
            loss,
            name2model[config["model_paths"][i]],
            name2tokenizer[config["tokenizer_paths"][i]],
            build_loss_args,
        )
    )
    lossfns[i].tokenizer.add_special_tokens({"mask_token": mlm_tokenizer.mask_token})
    loss2tokenizer[loss] = lossfns[i].tokenizer

# special_token_ids = mlm_tokenizer.convert_tokens_to_ids(mlm_tokenizer.all_special_tokens)
# intermediate_outputs = pd.read_json('outputs/toxicity/llm/bx3p1fwj/outputs_epsilon0.9.txt.intermediate', lines=True)
# intermediate_outputs = intermediate_outputs.explode('generations').reset_index(drop=True)
# intermediate_outputs_eda = intermediate_outputs.loc[intermediate_outputs['generations'].apply(len) != 0].reset_index(drop=True)
# intermediate_outputs_eda['prompt'] = intermediate_outputs_eda['prompt'].apply(lambda x: x['text'])
# intermediate_outputs_eda['masked_sentences'] = intermediate_outputs_eda['generations'].apply(lambda x: [item[1] for item in x.items() if 'mask' in item[0]])
# intermediate_outputs_eda = intermediate_outputs_eda.explode('masked_sentences').reset_index(drop=True)
all_source_texts = outputs_int_v1.loc[check_list, 'prompt'].tolist()
all_masked_sentences = outputs_int_v1.loc[check_list, 'masked_sentence'].tolist()
mask_info_dicts = []
span_lengths_es = []
for test_sent in all_masked_sentences:
    
    mask_info_dict, span_lengths = analyze_span_lengths_and_count(test_sent)
    mask_info_dicts.append(mask_info_dict)
    span_lengths_es.append(span_lengths)

source_text = all_source_texts[0]
test_sent = all_masked_sentences[0]
test_sent_span_lengths = span_lengths_es[0]

# merge masks
test_sent_merged = re.sub(r"(<mask>)+", "<mask>", test_sent)

# Max number of mask tokens to replace each span
max_mask_cnt_per_span = [max(x, config['max_tokens_per_span']) for x in test_sent_span_lengths]

# Get the span information of merged masks in the test sentence
mask_spans = [x.span() for x in re.finditer('<mask>',test_sent_merged)]

queue = []
queue.append(test_sent_merged[:mask_spans[0][0]])
# for i in range(len(mask_spans)):
i = 0 ### for now, we only consider the first span
curr_queue_size = len(queue)

# candidate generation
curr_full_text_hyp = [base_hyp + "<mask>" * max_mask_cnt_per_span[i] + test_sent_merged[mask_spans[i][1]:] for base_hyp in queue]
## Tokenize & conduct MLM inference
inputs = mlm_tokenizer(
    curr_full_text_hyp, return_tensors="pt", padding=True, truncation=True
)
inputs = inputs.to(config['device']) 
masked_sequence=inputs['input_ids']

if config['consider_prompt_for_cand_gen']:
    
    prompt_enc=mlm_tokenizer(mlm_tokenizer.bos_token + source_text,add_special_tokens=False, return_tensors="pt", padding=True, truncation=True).to(config['device'])
    prompt_enc['input_ids']=prompt_enc['input_ids'].expand(curr_queue_size,-1)
    prompt_enc['attention_mask']=prompt_enc['attention_mask'].expand(curr_queue_size,-1)
    # print(f"-- shape of source text after being tokenized and converted to input ids: {prompt_enc['input_ids'].shape}")
    
    input_tokens = torch.cat([prompt_enc.input_ids, inputs.input_ids], dim=1).to(config['device'])
    attention_masks = torch.cat([prompt_enc.attention_mask, inputs.attention_mask], dim=1).to(config['device'])
    
    # with torch.no_grad():
    #     logits = mlm(**inputs).logits
    with torch.no_grad():
        logits = mlm(input_ids = input_tokens, 
                    attention_mask = attention_masks).logits

    # Choose top k among non-special tokens
    # print(f"-- shape of logits before removing source text part: {logits.shape}")
    logits = logits[:, prompt_enc.input_ids.shape[1]:]
    
else:
    with torch.no_grad():
        logits = mlm(**inputs).logits

## Choose top k among non-special tokens
logits[:, :, special_token_ids] = -float("inf")

indices_in_mlm_tokens = (
    inputs.input_ids == mlm_tokenizer.mask_token_id
).nonzero(as_tuple=False) # if as_tuple=False, returns a tensor where column 1 indicates row indices, column 2 indicates column indices e.g. torch.Tensor([[0, 19],[0, 20], [0,38]])
# print(f"-- location of masks including masks in the future spans: {indices_in_mlm_tokens}")

## get post context -- added 12/10
post_context = test_sent_merged[mask_spans[i][1]:mask_spans[i+1][0]]

## For each hypothesis in curr_full_text_hyp, first max_mask_cnt_per_span[i] mask locations are relevant
indices_in_mlm_tokens = torch.cat([x[:max_mask_cnt_per_span[i]] for x in torch.chunk(indices_in_mlm_tokens, curr_queue_size)],dim=0)
indices_in_mlm_tokens_0 = indices_in_mlm_tokens[:,0]
indices_in_mlm_tokens_1 = indices_in_mlm_tokens[:,1]
# print(f"-- location of masks (row indices): {indices_in_mlm_tokens_0}")
# print(f"-- location of masks (col indices): {indices_in_mlm_tokens_1}")

## Get top k tokens for the j masks
predicted_token_ids = torch.topk(
    logits[indices_in_mlm_tokens_0, indices_in_mlm_tokens_1, :],
    k=config['k_per_location'],
    dim=-1,
)            

## beam search에 넣기 전에 이런 작업을 해주는게 좋을까? ## right side를 아예 안볼거면 ok.  -> 꼭 해주지 않아도 같은 결과가 나오긴 함.
print(f"masked_sequence: {masked_sequence}")
masked_sequence = [masked_sequence[ix, :indices_in_mlm_tokens_1[max_mask_cnt_per_span[i]*(ix+1)-1]+1] for ix in range(masked_sequence.shape[0])]
print(f"masked_sequence: {masked_sequence}")
masked_sequence = torch.nn.utils.rnn.pad_sequence(masked_sequence, batch_first=True, padding_value=mlm_tokenizer.pad_token_id)
print(f"masked_sequence: {masked_sequence}")

Starting new HTTPS connection (1): api.wandb.ai:443


https://api.wandb.ai:443 "POST /graphql HTTP/11" 200 None
https://api.wandb.ai:443 "POST /graphql HTTP/11" 200 None
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /roberta-large/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/tokenizer_config.json HTTP/11" 200 0
https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/config.json HTTP/11" 200 0


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-7B/resolve/main/generation_config.json HTTP/11" 200 0


OSError: Incorrect path_or_model_id: '/data/hyeryung/loc_edit/models/clean/roberta-base-jigsaw-toxicity-classifier-energy-training/step_1000_best_checkpoint'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
# 인자 설정
source_text = source_text
masked_sequence = masked_sequence
indices_in_mlm_tokens = (indices_in_mlm_tokens_0, indices_in_mlm_tokens_1)
predicted_token_ids = predicted_token_ids.indices
mlm_tokenizer = mlm_tokenizer
lossfns = lossfns
config = config

# 변수 초기화
final_hypotheses = [[] for i in range(len(masked_sequence))]
final_hypotheses_losses = [[] for i in range(len(masked_sequence))]
hypotheses = list(torch.split(masked_sequence,1,dim=0)) ## [torch.tensor([[a],[b],[c]]), torch.tensor([[d]])]
edit_indices = sorted(list(set(indices_in_mlm_tokens[1].tolist())))

print(f"hypotheses: {hypotheses}")
print(f"edit_indices: {edit_indices}")

loss_weights = config['loss_weights']

# deletion 케이스 처리 : deletion case로 final_hypotheses, final_hypotheses_losses 초기화 
first_mask_token_indices = [indices_in_mlm_tokens[1][indices_in_mlm_tokens[0]==i][0] for i in range(len(hypotheses))]
final_hypotheses = [[hypotheses[i][:, :first_mask_token_indices[i]].squeeze(0)] for i in range(len(hypotheses))] # List[List[torch.tensor]] # 계속해서 누적될 hypotheses 목록
tmp_hypotheses = [x[0].tolist() for x in final_hypotheses] # List[List[int]] # 이번 j 에서 고려할 hypotheses
print(f"{tmp_hypotheses}")

curr_loss = torch.zeros(len(tmp_hypotheses)).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
        
final_hypotheses_losses = [[x.squeeze(0)] for x in torch.split(curr_loss, 1)]
print(f"final_hypotheses_losses: {final_hypotheses_losses}")

In [ ]:
# for curr_edit_index in edit_indices:
curr_edit_index = edit_indices[0] ### no loop for now
    
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

print(f"batch_ids_to_edit: {batch_ids_to_edit}")
print(f"num_initial_hypotheses: {num_initial_hypotheses}")
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
print(f"tmp_hypotheses.shape: {tmp_hypotheses.shape}")
new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = new_func_candidates.to(config['device'])
print(f"new_func_candidates: {new_func_candidates}")
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"tmp_hypotheses: {tmp_hypotheses}")
# loss_weights = [1 - config['closs_weight'], config['closs_weight']]
loss_weights = config['loss_weights']
curr_loss = torch.zeros(tmp_hypotheses.shape[0]).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print("=== hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in hypotheses])
print("=== final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_hypotheses])
print("=== final_hypotheses_losses ===")
print(final_hypotheses_losses)

In [ ]:
curr_edit_index = edit_indices[1] ### 2nd mask 까지 decoding
    
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

print(f"batch_ids_to_edit: {batch_ids_to_edit}")
print(f"num_initial_hypotheses: {num_initial_hypotheses}")
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
print(f"tmp_hypotheses.shape: {tmp_hypotheses.shape}")
new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = new_func_candidates.to(config['device'])
print(f"new_func_candidates: {new_func_candidates}")
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"tmp_hypotheses: {tmp_hypotheses}")
# loss_weights = [1 - config['closs_weight'], config['closs_weight']]
loss_weights = config['loss_weights']
curr_loss = torch.zeros(tmp_hypotheses.shape[0]).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print("=== hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in hypotheses])
print("=== final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_hypotheses])
print("=== final_hypotheses_losses ===")
print(final_hypotheses_losses)

In [ ]:

curr_edit_index = edit_indices[2] ### 2nd mask 까지 decoding
    
batch_ids_to_edit = indices_in_mlm_tokens[0][indices_in_mlm_tokens[1]==curr_edit_index].tolist()
num_initial_hypotheses = [len(hypotheses[i]) for i in batch_ids_to_edit] ## keep track of initial hypotheses count e.g. [3, 1]
tmp_hypotheses = [hypotheses[i].repeat((config['k_per_location'],1)) for i in batch_ids_to_edit] ## [torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c]]), torch.tensor([[d],[d],[d]])]
num_initial_tmp_hypotheses = [len(x) for x in tmp_hypotheses]
tmp_hypotheses = torch.cat(tmp_hypotheses,dim=0) ## torch.tensor([[a],[b],[c],[a],[b],[c],[a],[b],[c],[d],[d],[d]])

print(f"batch_ids_to_edit: {batch_ids_to_edit}")
print(f"num_initial_hypotheses: {num_initial_hypotheses}")
print(f"num_initial_tmp_hypotheses: {num_initial_tmp_hypotheses}")
print(f"tmp_hypotheses.shape: {tmp_hypotheses.shape}")
new_func_candidates = predicted_token_ids[indices_in_mlm_tokens[1]==curr_edit_index] ## shape: (len(batch_ids_to_edit), k_per_location) e.g. [[x,y,z],[q,w,e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = repeat_interleave_unravel(new_func_candidates,num_initial_hypotheses) ## shape: (sum(num_initial_hypotheses), k_per_location) e.g. [[x],[x],[x],[y],[y],[y],[z],[z],[z],[q],[w],[e]]
print(f"new_func_candidates: {new_func_candidates}")
new_func_candidates = new_func_candidates.to(config['device'])
print(f"new_func_candidates: {new_func_candidates}")
tmp_hypotheses = torch.cat((tmp_hypotheses[ :, :curr_edit_index], new_func_candidates),dim=-1) ## tmp_hypotheses: [(a,b,c),(a,b,c), ..., (a,b,c)], new_func_candidates: [(p,p,p), (q,q,q), ..., (v,v,v)]
print(f"tmp_hypotheses: {tmp_hypotheses}")
# loss_weights = [1 - config['closs_weight'], config['closs_weight']]
loss_weights = config['loss_weights']
curr_loss = torch.zeros(tmp_hypotheses.shape[0]).to(config['device'])
for lossid, lossname in enumerate(config["losses"]):
    with torch.no_grad():
        lossvalue = lossfns[lossid].compute_gold_loss(
            source_text, mlm_tokenizer.batch_decode(tmp_hypotheses,skip_special_tokens=True),
            label_id=config['target_label_ids'][lossid],
        )
    torch.cuda.empty_cache()
    curr_loss += loss_weights[lossid] * lossvalue
print(f"curr_loss: {curr_loss}")
curr_loss = torch.split(curr_loss, num_initial_tmp_hypotheses, dim=0)
print(f"curr_loss: {curr_loss}")
top_beams = [torch.topk(x, k=config['beam_size'], dim=-1, largest=False).indices for x in curr_loss]
print(f"top_beams: {top_beams}")
tmp_hypotheses = torch.split(tmp_hypotheses, num_initial_tmp_hypotheses, dim=0)
print(f"tmp_hypotheses: {tmp_hypotheses}")
for jx, ix in enumerate(batch_ids_to_edit):
    hypotheses[ix] = torch.cat([tmp_hypotheses[jx][top_beams[jx]], masked_sequence[ix][curr_edit_index+1:].unsqueeze(0).repeat(config['beam_size'],1)], dim=-1)
    final_hypotheses[ix].extend(tmp_hypotheses[jx][top_beams[jx]])
    final_hypotheses_losses[ix].extend(curr_loss[jx][top_beams[jx]])
print("=== hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in hypotheses])
print("=== final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_hypotheses])
print("=== final_hypotheses_losses ===")
print(final_hypotheses_losses)

In [ ]:
final_top_beams = [torch.topk(torch.stack(x), k=config['beam_size'], dim=-1, largest=False).indices for x in final_hypotheses_losses]
final_final_hypotheses = [[x[i] for i in y] for x,y in zip(final_hypotheses,final_top_beams)]

In [ ]:
print("=== final_final_hypotheses ===")
print([mlm_tokenizer.batch_decode(x, skip_special_tokens=False) for x in final_final_hypotheses])
final_final_hypotheses_ = [torch.cat((x, masked_sequence[0, edit_indices[-1]+1:]), dim=-1) for x in final_final_hypotheses[0]]
print("=== final_final_hypotheses_ ===")
print([mlm_tokenizer.decode(x, skip_special_tokens=False) for x in final_final_hypotheses_])